# A2G-QFL: Conference vs Journal Midpoint-Manifold Comparison

This notebook version is converted from `A2G_QFL_conference_vs_journal_comparison.py`.

It keeps the **conference-version A2G** methods and adds the **journal-version midpoint-projected manifold A2G** methods for direct comparison:

- `FedAvg`
- `QoS_ONLY_Euclidean`
- `Conf_A2G_Euclidean`
- `Conf_A2G_Circular`
- `Journal_MP_A2G_QFL`
- `Journal_Adaptive_MP_A2G_QFL`

Before running on your local machine, activate your environment:

```powershell
conda activate ResearchAssistDeakin
jupyter notebook
```

For the Breast Lesions dataset on Windows, set the dataset path before launching Jupyter if needed:

```powershell
$env:LESIONS_CSV_PATH="C:\path\to\BrEaST-Lesions-USG-Clinical.csv"
jupyter notebook
```

You can also set a results folder:

```powershell
$env:RESULTS_DIR="C:\path\to\Results"
```


## Imports

In [1]:
from dataclasses import dataclass
from pathlib import Path
import csv
import os
import random
import time

import numpy as np
import matplotlib.pyplot as plt

from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_algorithms.optimizers import SPSA
from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_algorithms.utils import algorithm_globals
from qiskit_aer import Aer

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA

import pandas as pd
import torch

# OLD (deprecated)
# from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
# feature_map = ZZFeatureMap(feature_dimension=num_features, reps=2)
# ansatz = RealAmplitudes(num_qubits=num_features, reps=3)

# NEW
from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit import QuantumCircuit

from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

## 0. Teleportation config and link

In [2]:
# ============================================================
# 0. Teleportation config and link
# ============================================================

@dataclass
class TeleportCfg:
    use_teleportation: bool = True
    noise: str = "med"     # "low" | "med" | "high"
    noise_model: str = "bitflip"   # "bitflip" | "depolarizing" | "amp_damp"
    shots: int = 256        # samples to estimate fidelity

    # A2G QoS trust-gain exponents
    alpha: float = 0.0      # fidelity / quality emphasis
    gamma: float = 0.0      # latency penalty exponent
    delta: float = 0.0      # instability penalty exponent

    # A2G geometry gain. In the journal version this becomes beta_t
    # when adaptive_beta=True.
    beta: float = 1.0       # geometry gain (0=none, 1=full)

    # Journal-version midpoint-projected manifold aggregation controls
    adaptive_beta: bool = False   # if True, beta_t = max(beta_min, beta/(1+lambda*D_t))
    beta_min: float = 0.005       # lower bound for adaptive beta_t
    beta_lambda: float = 1.0      # sensitivity to manifold dispersion D_t
    max_tangent_norm: float = 0.0 # optional clipping of aggregated tangent vector; 0 disables

    seed: int = 2025

# simple bit-flip probability per noise regime
_NOISE_P = {
    "low": 0.01,
    "med": 0.06,
    "high": 0.12,
}

QOS_SIGNAL = "quantum_fidelity"   # or "classical_proxy"
GEOMETRY_MODE = "euclidean"       # "euclidean" | "circular" | "torus_midpoint"


#we are no longer limited to “synthetic bit-flip” only; now we can report depolarizing/amplitude damping results in a small table.
class TeleportationLink:
    """
    Toy but more realistic fidelity sampler for teleportation/entanglement quality.
    Adds depolarizing + amplitude-damping in addition to bit-flip.
    """
    def __init__(self, cfg: TeleportCfg):
        self.cfg = cfg
        self.rng = np.random.default_rng(cfg.seed)

    def sample_fidelity(self, shots=None, noise=None, noise_model=None):
        if shots is None:
            shots = self.cfg.shots
        if noise is None:
            noise = self.cfg.noise
        if noise_model is None:
            noise_model = self.cfg.noise_model

        p = float(_NOISE_P.get(noise, 0.06))
        shots = int(shots)

        if noise_model == "bitflip":
            # your original toy: with prob p, fidelity drops by random amount
            u = self.rng.uniform(0.5, 1.0, size=shots)
            flips = self.rng.binomial(1, p, size=shots)
            return np.clip(1.0 - flips * u, 0.0, 1.0)

        if noise_model == "depolarizing":
            # qubit depolarizing: E(ρ)=(1-p)ρ + p I/2 => mean state fidelity approx 1 - p/2
            mu = np.clip(1.0 - 0.5 * p, 0.0, 1.0)
            kappa = max(10.0, shots / 8.0)  # concentration
            a = max(1e-3, mu * kappa)
            b = max(1e-3, (1.0 - mu) * kappa)
            return self.rng.beta(a, b, size=shots)

        if noise_model == "amp_damp":
            # amplitude damping (toy): stronger degradation than depolarizing for same p
            # mean fidelity decreases roughly with p; we use mu = 1 - 0.7 p (clipped)
            mu = np.clip(1.0 - 0.7 * p, 0.0, 1.0)
            kappa = max(10.0, shots / 8.0)
            a = max(1e-3, mu * kappa)
            b = max(1e-3, (1.0 - mu) * kappa)
            return self.rng.beta(a, b, size=shots)

        raise ValueError(f"Unknown noise_model: {noise_model}")


def sample_packet_success_proxy(latency_sec: float, shots: int, rng: np.random.Generator, L_ref: float) -> tuple[float, float]:
    """
    Classical proxy: packet success probability decreases with latency.
    p_success = exp(-latency / L_ref). Then sample Bernoulli(shots).
    Returns (R_mean, R_var).
    """
    p_succ = float(np.exp(-max(latency_sec, 1e-6) / max(L_ref, 1e-6)))
    s = rng.binomial(1, p_succ, size=int(shots))
    return float(s.mean()), float(s.var())

## 1. A2G (Euclidean + QoS) aggregator

In [3]:
# ============================================================
# 1. A2G (Euclidean + QoS) aggregator
# ============================================================

#def agg_a2g_qos_euclidean(
   # local_ws,             # list of np.ndarray, one per client (θ_{i,t})
   # shard_sizes,          # list[int], |D_i|
   # fidelities,           # array-like, shape (K,)
   # latencies,            # array-like, shape (K,)
   # instabilities,        # array-like, shape (K,)
   # tele_cfg: TeleportCfg,
   # prev_global=None,     # np.ndarray or None (θ_t)
#):

#Now matched geometry agnostic

def _weight_entropy(W: np.ndarray) -> float:
    eps = 1e-12
    W = np.asarray(W, float)
    return float(-np.sum(W * np.log(W + eps)))


def _safe_norm(x: np.ndarray) -> float:
    return float(np.linalg.norm(np.asarray(x, float)))


def _maybe_clip_vector(v: np.ndarray, max_norm: float) -> tuple[np.ndarray, float, float]:
    """
    Clip a vector to max_norm if max_norm > 0.
    Returns (possibly clipped vector, original norm, clipped norm).
    """
    v = np.asarray(v, float)
    n0 = float(np.linalg.norm(v))
    if max_norm is not None and float(max_norm) > 0.0 and n0 > float(max_norm):
        v = v * (float(max_norm) / (n0 + 1e-12))
    n1 = float(np.linalg.norm(v))
    return v, n0, n1

def _direction_alignment(a: np.ndarray, b: np.ndarray, eps: float = 1e-12) -> float:
    """Cosine alignment between two tangent directions."""
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    return float(np.dot(a, b) / ((_safe_norm(a) * _safe_norm(b)) + eps))


#Add a helper for self-consistent midpoint update

def scm_torus_midpoint_update(
    A: np.ndarray,
    W: np.ndarray,
    prev_global: np.ndarray,
    beta0: float,
    max_tangent_norm: float = 0.0,
    scm_iters: int = 5,
    scm_tol: float = 1e-6,
    scm_solver_relax: float = 1.0,
):
    """
    Self-Consistent Midpoint Aggregation on the torus.

    Theory:
        u* = beta0 * sum_i W_i Log_{m(u*)}(theta_i)
        m(u*) = Exp_{theta_t}(0.5 * u*)

    Torus implementation:
        m(u) = wrap(theta_t + 0.5*u)
        Log_m(theta_i) = wrap(theta_i - m)
        theta_{t+1} = wrap(theta_t + u*)

    This does not reject a round. It finds an update whose own midpoint
    supports the same movement.
    """
    eps = 1e-12
    A = np.asarray(A, float)
    W = np.asarray(W, float)
    prev_global = np.asarray(prev_global, float)

    # Initial tangent directions from current global model.
    V0 = wrap_pi(A - prev_global[None, :])
    client_sq_dist = np.mean(V0 ** 2, axis=1)
    D0 = float(np.sum(W * client_sq_dist))

    # First QoS-weighted direction.
    v0 = np.sum(W[:, None] * V0, axis=0)
    v0, v0_norm_raw, v0_norm_clipped = _maybe_clip_vector(v0, max_tangent_norm)

    # Initial guess for the unknown self-consistent update.
    u = beta0 * v0

    last_psi = v0.copy()
    last_mid = wrap_pi(prev_global + 0.5 * u)
    converged = 0

    for r in range(int(scm_iters)):
        # Midpoint induced by the current update u.
        m = wrap_pi(prev_global + 0.5 * u)

        # Recompute client directions from this midpoint.
        V_mid = wrap_pi(A - m[None, :])
        psi = np.sum(W[:, None] * V_mid, axis=0)
        psi, psi_norm_raw, psi_norm_clipped = _maybe_clip_vector(
            psi, max_tangent_norm
        )

        # Fixed-point update: u = beta0 * psi(u)
        u_fp = beta0 * psi

        # Solver relaxation only controls numerical fixed-point solving.
        # scm_solver_relax=1.0 means pure fixed-point iteration.
        u_new = (1.0 - scm_solver_relax) * u + scm_solver_relax * u_fp

        fixed_point_change = _safe_norm(wrap_pi(u_new - u))

        u = u_new
        last_psi = psi
        last_mid = m

        if fixed_point_change < scm_tol:
            converged = 1
            break

    theta_raw = prev_global + u
    theta_next = wrap_pi(theta_raw)

    # Final residual of the self-consistency equation.
    scm_residual = _safe_norm(wrap_pi(u - beta0 * last_psi))

    # How much the implicit update moved compared with the first direction.
    implied_gain = float(_safe_norm(u) / (_safe_norm(v0) + eps))

    # Direction agreement between first direction and midpoint-supported direction.
    alignment = _direction_alignment(v0, last_psi)

    # Dispersion as seen from the self-consistent midpoint.
    V_final_mid = wrap_pi(A - last_mid[None, :])
    final_mid_sq = np.mean(V_final_mid ** 2, axis=1)
    D_mid = float(np.sum(W * final_mid_sq))

    diag_extra = {
        "scm_iters": int(r + 1),
        "scm_converged": int(converged),
        "scm_residual": float(scm_residual),
        "scm_implied_gain": float(implied_gain),
        "scm_direction_alignment": float(alignment),
        "scm_midpoint_dispersion": float(D_mid),
        "scm_beta0": float(beta0),
        "scm_solver_relax": float(scm_solver_relax),
        "midpoint_shift_norm": _safe_norm(wrap_pi(last_mid - prev_global)),
        "update_norm": _safe_norm(wrap_pi(theta_next - prev_global)),
        "projection_shift_norm": _safe_norm(theta_raw - theta_next),
        "tangent_norm": float(v0_norm_raw),
        "tangent_norm_clipped": float(v0_norm_clipped),
        "v_mid_norm": _safe_norm(last_psi),
    }

    return theta_next, diag_extra

def agg_a2g_qos(
    local_ws, shard_sizes,
    quality_means, latencies, quality_vars,
    tele_cfg: TeleportCfg,
    prev_global=None,
    geometry: str = "euclidean",   # "euclidean" | "circular" | "torus_midpoint"
):
    """
    A2G aggregation with journal-version manifold geometry option.

    Existing conference-style modes
    --------------------------------
    geometry="euclidean":
        QoS-weighted Euclidean aggregate followed by beta-relaxed movement.

    geometry="circular":
        QoS-weighted coordinate-wise circular aggregate followed by wrapped
        beta-relaxed movement.

    Journal-version mode
    --------------------
    geometry="torus_midpoint":
        QoS-weighted midpoint-projected manifold aggregation on the product
        torus T^d. This treats each QNN rotation parameter as periodic:
            theta_j == theta_j + 2*pi.

        The server computes client-wise tangent directions around theta_t,
        forms a QoS-weighted tangent direction, evaluates a midpoint, recomputes
        tangent directions at the midpoint, and projects/wraps the final update.
    """
    local_ws = [np.asarray(w, float) for w in local_ws]
    A = np.stack(local_ws, axis=0)  # (K, D)
    K, Ddim = A.shape

    shard_sizes = np.asarray(shard_sizes, float)
    shard_sizes = np.maximum(shard_sizes, 1.0)
    p_i = shard_sizes / (shard_sizes.sum() + 1e-12)

    Q = np.clip(np.asarray(quality_means, float), 0.0, 1.0)      # F_mean or proxy reliability
    L = np.maximum(np.asarray(latencies, float), 1e-6)
    V = np.maximum(np.asarray(quality_vars, float), 0.0)

    alpha = float(getattr(tele_cfg, "alpha", 1.0))
    gamma = float(getattr(tele_cfg, "gamma", 1.0))
    delta = float(getattr(tele_cfg, "delta", 1.0))
    beta_raw = float(getattr(tele_cfg, "beta", 1.0))
    adaptive_beta_flag = bool(getattr(tele_cfg, "adaptive_beta", False))
    beta_min = float(getattr(tele_cfg, "beta_min", 0.005))
    beta_lambda = float(getattr(tele_cfg, "beta_lambda", 1.0))
    max_tangent_norm = float(getattr(tele_cfg, "max_tangent_norm", 0.0))

    # QoS trust score and normalized client aggregation weights
    q = (Q ** alpha) / ((L ** gamma) * ((V + 1e-8) ** delta))
    w_unnorm = p_i * q
    W = p_i if w_unnorm.sum() <= 0 else (w_unnorm / (w_unnorm.sum() + 1e-12))

    # Default diagnostics; methods below overwrite relevant fields.
    diag = {
        "beta": beta_raw,
        "beta_raw": beta_raw,
        "beta_eff": beta_raw,
        "adaptive_beta": int(adaptive_beta_flag),
        "geometry": geometry,
        "mean_weight": float(W.mean()),
        "min_weight": float(W.min()),
        "max_weight": float(W.max()),
        "weight_entropy": _weight_entropy(W),
        "manifold_dispersion": 0.0,
        "tangent_norm": 0.0,
        "tangent_norm_clipped": 0.0,
        "midpoint_shift_norm": 0.0,
        "projection_shift_norm": 0.0,
        "update_norm": 0.0,
        "theta_bar_norm": 0.0,
    }

    # ------------------------------------------------------------
    # Baseline / conference-version aggregation modes
    # ------------------------------------------------------------
    if geometry == "euclidean":
        theta_bar = weighted_euclidean_mean(local_ws, W)
        if prev_global is None:
            theta_next = theta_bar
        else:
            prev_global = np.asarray(prev_global, float)
            raw_update = theta_bar - prev_global
            theta_next = prev_global + beta_raw * raw_update
            diag["update_norm"] = _safe_norm(theta_next - prev_global)
            diag["manifold_dispersion"] = float(np.sum(W * np.mean((A - prev_global[None, :]) ** 2, axis=1)))
        diag["theta_bar_norm"] = _safe_norm(theta_bar)
        return theta_next, diag, W

    if geometry == "circular":
        theta_bar = weighted_circular_mean(local_ws, W)
        if prev_global is None:
            theta_next = theta_bar
        else:
            prev_global = np.asarray(prev_global, float)
            raw_update = wrap_pi(theta_bar - prev_global)
            theta_next = wrap_pi(prev_global + beta_raw * raw_update)
            diag["update_norm"] = _safe_norm(wrap_pi(theta_next - prev_global))
            diag["manifold_dispersion"] = float(np.sum(W * np.mean(wrap_pi(A - prev_global[None, :]) ** 2, axis=1)))
        diag["theta_bar_norm"] = _safe_norm(theta_bar)
        return theta_next, diag, W

    # ------------------------------------------------------------
    # Journal-version method: midpoint-projected manifold A2G-QFL
    # ------------------------------------------------------------
    if geometry == "torus_midpoint":
        # If there is no previous global parameter, initialize on the manifold
        # using the QoS-weighted circular target.
        theta_circ = weighted_circular_mean(local_ws, W)
        if prev_global is None:
            theta_next = theta_circ
            diag["theta_bar_norm"] = _safe_norm(theta_circ)
            return theta_next, diag, W

        prev_global = np.asarray(prev_global, float)

        # 1) Client-wise Log map at current global model:
        #    v_i = Log_{theta_t}(theta_i) = wrap(theta_i - theta_t)
        V_tangent = wrap_pi(A - prev_global[None, :])  # (K, D)

        # Scale-normalized manifold dispersion, useful across circuits with
        # different parameter dimensions.
        client_sq_dist = np.mean(V_tangent ** 2, axis=1)
        D_t = float(np.sum(W * client_sq_dist))

        # 2) Adaptive geometry gain beta_t if enabled.
        if adaptive_beta_flag:
            beta_eff = float(max(beta_min, beta_raw / (1.0 + beta_lambda * D_t)))
        else:
            beta_eff = beta_raw

        # 3) QoS-weighted tangent aggregation at theta_t.
        v_t = np.sum(W[:, None] * V_tangent, axis=0)
        v_t, tangent_norm_raw, tangent_norm_clipped = _maybe_clip_vector(v_t, max_tangent_norm)

        # 4) Midpoint evaluation on the torus.
        theta_mid = wrap_pi(prev_global + 0.5 * beta_eff * v_t)

        # 5) Recompute client-wise Log maps at midpoint.
        V_mid = wrap_pi(A - theta_mid[None, :])
        v_mid = np.sum(W[:, None] * V_mid, axis=0)
        v_mid, v_mid_norm_raw, v_mid_norm_clipped = _maybe_clip_vector(v_mid, max_tangent_norm)

        # 6) Final projected/wrapped update from theta_t using midpoint-corrected direction.
        theta_raw = prev_global + beta_eff * v_mid
        theta_next = wrap_pi(theta_raw)

        diag.update({
            "beta": beta_eff,                 # for backward-compatible plots
            "beta_raw": beta_raw,
            "beta_eff": beta_eff,
            "adaptive_beta": int(adaptive_beta_flag),
            "manifold_dispersion": D_t,
            "tangent_norm": tangent_norm_raw,
            "tangent_norm_clipped": tangent_norm_clipped,
            "midpoint_shift_norm": _safe_norm(wrap_pi(theta_mid - prev_global)),
            "projection_shift_norm": _safe_norm(theta_raw - theta_next),
            "update_norm": _safe_norm(wrap_pi(theta_next - prev_global)),
            "theta_bar_norm": _safe_norm(theta_circ),
            "v_mid_norm": v_mid_norm_raw,
            "v_mid_norm_clipped": v_mid_norm_clipped,
        })
        return theta_next, diag, W

        # ------------------------------------------------------------
    # Proposed method: Self-Consistent Midpoint A2G-QFL
    # ------------------------------------------------------------
    if geometry == "torus_scm_midpoint":
        # First round: no previous global point exists.
        # Use QoS-weighted circular initialization on the torus.
        theta_circ = weighted_circular_mean(local_ws, W)

        if prev_global is None:
            theta_next = theta_circ
            diag.update({
                "theta_bar_norm": _safe_norm(theta_circ),
                "scm_iters": 0,
                "scm_converged": 1,
                "scm_residual": 0.0,
                "scm_implied_gain": 0.0,
                "scm_direction_alignment": 1.0,
                "scm_midpoint_dispersion": 0.0,
            })
            return theta_next, diag, W

        prev_global = np.asarray(prev_global, float)

        # Log maps at current global point for diagnostics and optional adaptive beta.
        V_tangent = wrap_pi(A - prev_global[None, :])
        client_sq_dist = np.mean(V_tangent ** 2, axis=1)
        D_t = float(np.sum(W * client_sq_dist))

        # beta_raw is now interpreted as beta0 in the self-consistency equation:
        # u* = beta0 * psi(u*)
        #
        # For the cleanest theoretical version, set adaptive_beta=False.
        # If adaptive_beta=True, beta0 is additionally moderated by D_t.
        if adaptive_beta_flag:
            beta_eff = float(max(beta_min, beta_raw / (1.0 + beta_lambda * D_t)))
        else:
            beta_eff = beta_raw

        scm_iters = int(getattr(tele_cfg, "scm_iters", 5))
        scm_tol = float(getattr(tele_cfg, "scm_tol", 1e-6))
        scm_solver_relax = float(getattr(tele_cfg, "scm_solver_relax", 1.0))

        theta_next, scm_diag = scm_torus_midpoint_update(
            A=A,
            W=W,
            prev_global=prev_global,
            beta0=beta_eff,
            max_tangent_norm=max_tangent_norm,
            scm_iters=scm_iters,
            scm_tol=scm_tol,
            scm_solver_relax=scm_solver_relax,
        )

        diag.update({
            "beta": beta_eff,
            "beta_raw": beta_raw,
            "beta_eff": beta_eff,
            "adaptive_beta": int(adaptive_beta_flag),
            "manifold_dispersion": D_t,
            "theta_bar_norm": _safe_norm(theta_circ),
        })
        diag.update(scm_diag)

        return theta_next, diag, W

    raise ValueError(
        "geometry must be 'euclidean', 'circular', 'torus_midpoint', "
        "or 'torus_scm_midpoint'"
    )

## 2. Telemetry helpers

In [4]:
# ============================================================
# 2. Telemetry helpers
# ============================================================

#def measure_instability(weights_vec: np.ndarray) -> float:
    #return float(np.var(weights_vec))
# above is parameter variance but this shoould be ;Qos instability,
def param_variance(weights_vec: np.ndarray) -> float:
    """Optional diagnostic: variance of the parameter vector."""
    return float(np.var(weights_vec))

def extract_client_weights(model) -> np.ndarray:
    """Extract trainable weights from NeuralNetworkClassifier(SamplerQNN)."""
    if hasattr(model, "weights") and model.weights is not None:
        return np.asarray(model.weights, float)
    qnn = getattr(model, "neural_network", None)
    if qnn is not None and hasattr(qnn, "weights"):
        return np.asarray(qnn.weights, float)
    raise RuntimeError("Cannot extract weights from model.")

def append_round_telemetry(path, epoch, global_acc,
                           fidelities, latencies, instabilities,
                           weights, diag):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    header = [
        "epoch","global_acc",
        "fid_mean","fid_std",
        "lat_mean","lat_p90",
        "instab_mean",
        "weight_entropy",
        "beta","mean_weight","min_weight","max_weight",
    ]
    write_header = not path.exists()
    with open(path, "a", newline="") as f:
        w = csv.writer(f)
        if write_header:
            w.writerow(header)
        eps = 1e-12
        ent = float(-np.sum(weights * np.log(weights + eps)))
        row = [
            int(epoch), float(global_acc),
            float(np.mean(fidelities)), float(np.std(fidelities)),
            float(np.mean(latencies)), float(np.percentile(latencies, 90)),
            float(np.mean(instabilities)),
            ent,
            float(diag.get("beta", np.nan)),
            float(diag.get("mean_weight", np.nan)),
            float(diag.get("min_weight", np.nan)),
            float(diag.get("max_weight", np.nan)),
        ]
        w.writerow(row)

def append_client_telemetry(path, epoch, per_client_rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    header = [
    "epoch","client_id","shard_size",
    "train_acc","test_acc",
    "train_loss","test_loss",
    "latency_sec","fidelity_mean",
    "instability_var","param_var","trust_weight"
]
    write_header = not path.exists()
    with open(path, "a", newline="") as f:
        w = csv.writer(f)
        if write_header:
            w.writerow(header)
        for r in per_client_rows:
            w.writerow([
            int(epoch), int(r["client_id"]), int(r["shard_size"]),
            float(r["train_acc_local"]), float(r["test_acc_local"]),
            float(r["train_loss_local"]), float(r["test_loss_local"]),
            float(r["time_sec_local"]), float(r["fidelity_mean"]),
            float(r["instability_var"]), float(r["param_var"]),
            float(r["trust_weight"]),
        ])

def init_metrics_csv(csv_path: str, num_clients: int):
    """Create/overwrite CSV with a header for epoch/global/client metrics."""
    path = Path(csv_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    header = ["Epoch", "GlobalAccuracy"]
    for i in range(num_clients):
        header += [f"Client{i}_TrainAcc", f"Client{i}_TestAcc"]
    with open(path, "w", newline="") as f:
        csv.writer(f).writerow(header)

def append_metrics_row(csv_path: str, epoch: int, global_acc: float,
                       train_accs, test_accs):
    """Append one epoch’s metrics."""
    path = Path(csv_path)
    with open(path, "a", newline="") as f:
        w = csv.writer(f)
        row = [epoch, global_acc]
        for tr, te in zip(train_accs, test_accs):
            row.extend([tr, te])
        w.writerow(row)

## 3. QNN model construction

In [5]:
# ============================================================
# 3. QNN model construction
# ============================================================
LOCAL_SPSA_MAXITER = 5   # smoke test: 3–5, final run: 10–25

def create_qnn_model(num_features: int, initial_point=None):
    feature_map = zz_feature_map(feature_dimension=num_features, reps=2)
    ansatz = real_amplitudes(num_qubits=num_features, reps=3)

    qc = QuantumCircuit(num_features)
    qc.compose(feature_map, inplace=True)
    qc.compose(ansatz, inplace=True)

    def parity(x):
        return "{:b}".format(x).count("1") % 2

    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

    sampler_qnn = SamplerQNN(
        circuit=qc,
        interpret=parity,
        output_shape=2,
        input_params=feature_map.parameters,
        weight_params=ansatz.parameters,
        pass_manager=pm,
    )

    classifier = NeuralNetworkClassifier(
        neural_network=sampler_qnn,
        optimizer=SPSA(LOCAL_SPSA_MAXITER),
        initial_point=None if initial_point is None else np.asarray(initial_point, float),
        warm_start=False,
    )

    return classifier

def train_qnn_model(
    X_train_data, y_train_data,
    X_test_data, y_test_data,
    model=None,
    eval_local=False,
):
    if model is None:
        num_features = X_train_data.shape[1]
        model = create_qnn_model(num_features)

    start_time = time.time()
    model.fit(X_train_data, y_train_data)
    elapsed_time = time.time() - start_time

    if eval_local:
        train_score = model.score(X_train_data, y_train_data)
        test_score = model.score(X_test_data, y_test_data)
    else:
        train_score = np.nan
        test_score = np.nan

    return model, train_score, test_score, elapsed_time

## 4. Global accuracy from averaged weights

In [6]:
# ============================================================
# 4. Global accuracy from averaged weights
# ============================================================

def _logits_to_labels(y_raw: np.ndarray) -> np.ndarray:
    """Robust conversion from network outputs to class labels."""
    y_raw = np.asarray(y_raw)
    if y_raw.ndim == 1:
        return (y_raw >= 0).astype(int)
    elif y_raw.ndim == 2:
        if y_raw.shape[1] == 1:
            return (y_raw[:, 0] >= 0).astype(int)
        return np.argmax(y_raw, axis=1)
    else:
        raise ValueError(f"Unexpected network output shape: {y_raw.shape}")

def compute_global_accuracy_from_weights(prototype_model, avg_weights: np.ndarray,
                                         X_test, y_test) -> float:
    """Forward the averaged parameter vector through the QNN and compute accuracy."""
    qnn = getattr(prototype_model, "neural_network", None)
    if qnn is None:
        qnn = getattr(prototype_model, "_neural_network", None)
    if qnn is None:
        raise RuntimeError("Cannot access underlying QNN from classifier.")

    y_raw = qnn.forward(X_test, np.asarray(avg_weights))
    y_pred = _logits_to_labels(y_raw)
    return float(np.mean(y_pred == y_test))

## 5. Data + client setup (3 datasets)

In [7]:
# ============================================================
# 5. Data + client setup (3 datasets)
# ============================================================

class Client:
    def __init__(self, train_data):
        self.client_train_data = train_data  # (X, y)
        self.models = []
        self.train_scores = []
        self.test_scores = []
        self.primary_model = None

from sklearn.model_selection import train_test_split


# --- helpers for lesions data non-IID splits ---


def assign_label_skewed_data(X, y, num_clients, classes_per_client=1):
    """Assign non-IID data by restricting clients to certain labels."""
    client_data = [[] for _ in range(num_clients)]
    client_labels = [[] for _ in range(num_clients)]

    label_to_indices = {label: np.where(y == label)[0] for label in np.unique(y)}

    for indices in label_to_indices.values():
        np.random.shuffle(indices)

    class_partitions = np.array_split(np.unique(y), num_clients // classes_per_client)

    client_id = 0
    for class_subset in class_partitions:
        for _ in range(classes_per_client):
            if client_id >= num_clients:
                break
            client_indices = []
            for c in class_subset:
                count = len(label_to_indices[c]) // classes_per_client
                client_indices.extend(label_to_indices[c][:count])
                label_to_indices[c] = label_to_indices[c][count:]
            np.random.shuffle(client_indices)
            client_data[client_id] = X[client_indices]
            client_labels[client_id] = y[client_indices]
            client_id += 1

    return client_data, client_labels

def assign_quantity_skewed_data(X, y, num_clients, min_size=10, max_size=100):
    """
    Assign non-IID data by giving each client a random amount of data.
    """
    total_data = len(X)
    client_data, client_labels = [], []

    remaining_indices = np.arange(total_data)
    np.random.shuffle(remaining_indices)

    current_index = 0
    for i in range(num_clients):
        client_size = np.random.randint(min_size, max_size + 1)
        if current_index + client_size > total_data:
            client_size = total_data - current_index
        selected_indices = remaining_indices[current_index: current_index + client_size]
        current_index += client_size

        client_data.append(X[selected_indices])
        client_labels.append(y[selected_indices])

        if current_index >= total_data:
            break

    while len(client_data) < num_clients:
        rand_idx = np.random.randint(0, len(client_data))
        client_data.append(client_data[rand_idx].copy())
        client_labels.append(client_labels[rand_idx].copy())

    return client_data, client_labels

import numpy as np

def assign_label_skewed_dirichlet(X, y, num_clients, alpha=0.3, min_size=10, seed=42):
    """
    Non-IID label-skew partition using a Dirichlet distribution.
    - Every class is present globally.
    - Each client gets a different label proportion (skew).
    - Ensures each client has at least `min_size` samples (by topping up if needed).
    """
    rng = np.random.default_rng(seed)
    X = np.asarray(X)
    y = np.asarray(y)

    labels = np.unique(y)
    # indices of each label
    idx_by_label = {lbl: np.where(y == lbl)[0] for lbl in labels}
    for lbl in labels:
        rng.shuffle(idx_by_label[lbl])

    client_indices = [[] for _ in range(num_clients)]

    # For each label, distribute its samples across clients with Dirichlet proportions
    for lbl in labels:
        idxs = idx_by_label[lbl]
        n_lbl = len(idxs)
        if n_lbl == 0:
            continue

        # Dirichlet over clients for this label
        proportions = rng.dirichlet(alpha * np.ones(num_clients))
        # initial integer allocation
        counts = np.floor(proportions * n_lbl).astype(int)

        # fix rounding so that sum(counts) == n_lbl
        diff = n_lbl - counts.sum()
        if diff > 0:
            # give the leftover samples to clients with largest proportions
            for cid in np.argsort(proportions)[-diff:]:
                counts[cid] += 1
        elif diff < 0:
            # remove extra samples from clients with largest counts
            for cid in np.argsort(counts)[-(-diff):]:
                if counts[cid] > 0:
                    counts[cid] -= 1

        # now distribute actual indices
        start = 0
        for cid in range(num_clients):
            c = counts[cid]
            if c > 0:
                client_indices[cid].extend(idxs[start:start + c])
                start += c

    # Ensure every client has at least `min_size` samples
    all_indices = np.arange(len(X))
    for cid in range(num_clients):
        if len(client_indices[cid]) < min_size:
            needed = min_size - len(client_indices[cid])
            extra = rng.choice(all_indices, size=needed, replace=False)
            client_indices[cid].extend(extra.tolist())

    # Build final per-client datasets
    client_data = [X[np.array(idxs)] for idxs in client_indices]
    client_labels = [y[np.array(idxs)] for idxs in client_indices]
    return client_data, client_labels

# -----------------------------
# choose dataset here
# -----------------------------
DATASET = "lesions"   # options: "sk_breast", "genome", "lesions", "baf"
num_clients = 5

# Global seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
algorithm_globals.random_seed = SEED
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)

backend = Aer.get_backend("aer_simulator")

##################New camera ready##########
def wrap_pi(x):
    return (x + np.pi) % (2*np.pi) - np.pi

def weighted_euclidean_mean(local_ws, W):
    A = np.stack([np.asarray(w, float) for w in local_ws], axis=0)
    return np.sum(W[:, None] * A, axis=0)


#def weighted_circular_mean(local_ws, W):
   # A = np.stack([np.asarray(w, float) for w in local_ws], axis=0)
    #z = np.sum(W[:, None] * np.exp(1j * A), axis=0)
    #return np.angle(z)  # in (-pi, pi]

def weighted_circular_mean(local_ws, W):
    A = np.stack([np.asarray(w, float) for w in local_ws], axis=0)  # (K, D)
    W = np.asarray(W, float)[:, None]                               # (K, 1)
    s = np.sum(W * np.sin(A), axis=0)
    c = np.sum(W * np.cos(A), axis=0)
    return np.arctan2(s, c)
##################New camera ready##########

#becasue from baf preprocess file return reords not x, y
def records_to_xy(records):
    """
    Convert records returned by preprocess_baf.py into NumPy arrays.

    Each record has:
        record["sequence"] -> feature vector
        record["label"]    -> class label
    """
    X = np.stack([r["sequence"] for r in records]).astype(np.float32)
    y = np.asarray([r["label"] for r in records], dtype=int)
    return X, y


# --- Dataset 1: sklearn Breast Cancer (original baseline) ---

if DATASET == "sk_breast":
    data = load_breast_cancer()
    X = data.data
    y = data.target

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    pca = PCA(n_components=4)
    X = pca.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # simple random equal splits across clients
    indices = np.arange(X_train.shape[0])
    np.random.shuffle(indices)
    splits = np.array_split(indices, num_clients)

    clients = []
    for split in splits:
        clients.append(Client((X_train[split], y_train[split])))

    global_test_data = (X_test, y_test)

# --- Dataset 2: Genomic Benchmarks DemoHumanOrWorm ---
elif DATASET == "genome":
    from genomic_benchmarks.dataset_getters.pytorch_datasets import DemoHumanOrWorm

    train_set = DemoHumanOrWorm(split='train', version=0)
    test_set  = DemoHumanOrWorm(split='test',  version=0)

    word_size = 40

    # build dictionary from train_set only
    from collections import defaultdict
    word_combinations = {}
    iteration = 1
    for text, _ in train_set:
        text = text.strip()
        for i in range(0, len(text), word_size):
            word = text[i:i+word_size]
            if word not in word_combinations:
                word_combinations[word] = iteration
                iteration += 1

    def encode_dataset(dataset, word_dict, word_size=40):
        seqs = []
        labels = []
        for text, label in dataset:
            text = text.strip()
            words = [text[i:i+word_size] for i in range(0, len(text), word_size)]
            int_seq = [word_dict[w] for w in words if w in word_dict]
            seqs.append(np.array(int_seq, dtype=float))
            labels.append(int(label))
        # pad to common length
        max_len = max(len(s) for s in seqs)
        X_arr = np.zeros((len(seqs), max_len), dtype=float)
        for i, s in enumerate(seqs):
            X_arr[i, :len(s)] = s
        y_arr = np.array(labels, dtype=int)
        return X_arr, y_arr

    X_train_raw, y_train = encode_dataset(train_set, word_combinations, word_size)
    X_test_raw,  y_test  = encode_dataset(test_set,  word_combinations, word_size)

    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test  = scaler.transform(X_test_raw)

    # shuffle train set before splitting among clients
    rng = np.random.default_rng(42)
    perm = rng.permutation(len(X_train))
    X_train = X_train[perm]
    y_train = y_train[perm]

    indices = np.arange(X_train.shape[0])
    np.random.shuffle(indices)
    splits = np.array_split(indices, num_clients)

    clients = []
    for split in splits:
        clients.append(Client((X_train[split], y_train[split])))

    global_test_data = (X_test, y_test)

# --- Dataset 3: Breast Lesions CSV (non-IID, quantity-skewed) ---
elif DATASET == "lesions":
    # path may need adjustment for your environment
    csv_path = os.getenv("LESIONS_CSV_PATH", "BrEaST-Lesions-USG-Clinical.csv")
    df = pd.read_csv(csv_path)

    selected_features = ["Age", "Shape", "Echogenicity",
                         "Posterior_features", "Calcifications", "Classification"]
    df = df[selected_features]

    df = df[
        (df["Age"] != "not available") &
        (~df["Shape"].isin(["not applicable"])) &
        (~df["Echogenicity"].isin(["not applicable"])) &
        (~df["Posterior_features"].isin(["not applicable"])) &
        (~df["Calcifications"].isin(["not applicable", "indefinable"])) &
        (df["Classification"].isin(["benign", "malignant"]))
    ].copy()

    df["Age"] = pd.to_numeric(df["Age"])

    from sklearn.preprocessing import LabelEncoder
    for col in ["Shape", "Echogenicity", "Posterior_features", "Calcifications"]:
        df[col] = LabelEncoder().fit_transform(df[col])

    df["Label"] = df["Classification"].map({"benign": 0, "malignant": 1})
    df.drop(columns=["Classification"], inplace=True)

    X = df.drop(columns=["Label"]).values
    y = df["Label"].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA(n_components=5)
    X_pca = pca.fit_transform(X_scaled)

    X_train, X_test, y_train, y_test = train_test_split(
        X_pca, y, test_size=0.3, random_state=42, stratify=y
    )

    #client_train_data, client_train_labels = assign_quantity_skewed_data(
    #X_train, y_train, num_clients, min_size=20, max_size=150
#)

    # non-IID label skew across clients
    client_train_data, client_train_labels = assign_label_skewed_dirichlet(
    X_train, y_train,
    num_clients=num_clients,
    alpha=0.3,      # smaller → stronger label skew
    min_size=20,    # guarantee at least 20 samples per client
    seed=42
)


    clients = []
    for k in range(num_clients):
        clients.append(Client((client_train_data[k], client_train_labels[k])))

    global_test_data = (X_test, y_test)

# --- Dataset 4: Bank Account Fraud dataset ---
elif DATASET == "baf":
    from preprocess_baf import load_and_prepare_dataset as load_baf_dataset

    # Change this path to your actual BAF CSV file
    baf_csv_path = os.getenv(
        "BAF_CSV_PATH",
        "Base.csv"   # example: replace with your real CSV name/path
    )

    train_records, val_records, test_records = load_baf_dataset(
        csv_path=baf_csv_path,
        n_features=4,              # should match QNN feature dimension / qubits
        global_seed=SEED,
        target_col="fraud_bool",
        test_size=0.25,
        val_size=0.15,
        max_samples=5000,          # increase if runtime allows 100000
        scale_to_angles=True       # useful for angle-based quantum encoding
    )

    X_train, y_train = records_to_xy(train_records)
    X_val, y_val = records_to_xy(val_records)
    X_test, y_test = records_to_xy(test_records)

    # Non-IID client split for federated setting
    client_train_data, client_train_labels = assign_label_skewed_dirichlet(
        X_train,
        y_train,
        num_clients=num_clients,
        alpha=0.3,
        min_size=20,
        seed=SEED
    )

    clients = []
    for k in range(num_clients):
        clients.append(Client((client_train_data[k], client_train_labels[k])))

    global_val_data = (X_val, y_val)
    global_test_data = (X_test, y_test)

else:
    raise ValueError(f"Unknown DATASET: {DATASET}")

# --------------------------------------------------
# Now split global test into validation and final test
# --------------------------------------------------
from sklearn.model_selection import train_test_split

# --------------------------------------------------
# Now split global test into validation and final test
# Only needed for datasets that do not already provide validation data.
# BAF already returns train/val/test from preprocess_baf.py.
# --------------------------------------------------

if DATASET != "baf":
    X_global_test_full, y_global_test_full = global_test_data

    X_global_val, X_global_test, y_global_val, y_global_test = train_test_split(
        X_global_test_full,
        y_global_test_full,
        test_size=0.5,
        random_state=2025,
        stratify=y_global_test_full
    )

    global_val_data = (X_global_val, y_global_val)
    global_test_data = (X_global_test, y_global_test)
#=================================================
#Add baseline

#===================================================
def coord_median(local_ws):
    A = np.stack([np.asarray(w, float) for w in local_ws], axis=0)  # (K,D)
    return np.median(A, axis=0)

def trimmed_mean(local_ws, trim_ratio=0.2):
    A = np.stack([np.asarray(w, float) for w in local_ws], axis=0)  # (K,D)
    K = A.shape[0]
    r = int(np.floor(trim_ratio * K))
    if 2*r >= K:
        return np.mean(A, axis=0)
    A_sorted = np.sort(A, axis=0)
    return np.mean(A_sorted[r:K-r, :], axis=0)
#--------------------------------------------------
#Helpers for saving results
#-------------------------------------------------


from pathlib import Path
import csv
import os
import numpy as np

def init_round_csv(path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="") as f:
        csv.writer(f).writerow([
            "epoch", "global_acc",
            "qos_mean", "qos_std", "qos_var_mean",
            "lat_mean", "lat_p90",
            "weight_entropy",
            "beta", "beta_raw", "beta_eff", "adaptive_beta",
            "geometry", "qos_signal", "noise", "noise_model",
            "mean_weight", "min_weight", "max_weight",
            # Journal-version geometry diagnostics
            "manifold_dispersion", "tangent_norm", "tangent_norm_clipped",
            "midpoint_shift_norm", "projection_shift_norm", "update_norm",
            "theta_bar_norm", "v_mid_norm", "v_mid_norm_clipped",
        ])


def append_round_csv(path: Path, epoch: int, global_acc: float,
                     qos_means, latencies, qos_vars,
                     agg_w, diag, qos_signal: str, noise: str, noise_model: str):
    eps = 1e-12
    ent = float(-np.sum(agg_w * np.log(agg_w + eps)))
    row = [
        int(epoch), float(global_acc),
        float(np.mean(qos_means)), float(np.std(qos_means)), float(np.mean(qos_vars)),
        float(np.mean(latencies)), float(np.percentile(latencies, 90)),
        ent,
        float(diag.get("beta", np.nan)),
        float(diag.get("beta_raw", np.nan)),
        float(diag.get("beta_eff", np.nan)),
        int(diag.get("adaptive_beta", 0)),
        str(diag.get("geometry", "")),
        qos_signal, noise, noise_model,
        float(diag.get("mean_weight", np.nan)),
        float(diag.get("min_weight", np.nan)),
        float(diag.get("max_weight", np.nan)),
        float(diag.get("manifold_dispersion", np.nan)),
        float(diag.get("tangent_norm", np.nan)),
        float(diag.get("tangent_norm_clipped", np.nan)),
        float(diag.get("midpoint_shift_norm", np.nan)),
        float(diag.get("projection_shift_norm", np.nan)),
        float(diag.get("update_norm", np.nan)),
        float(diag.get("theta_bar_norm", np.nan)),
        float(diag.get("v_mid_norm", np.nan)),
        float(diag.get("v_mid_norm_clipped", np.nan)),
    ]
    with open(path, "a", newline="") as f:
        w = csv.writer(f)
        w.writerow(row)
        f.flush()


def init_client_csv(path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="") as f:
        csv.writer(f).writerow([
            "epoch","client_id","shard_size",
            "latency_sec","qos_mean","qos_var",
            "param_var","trust_weight",
            "train_acc_local","test_acc_local"
        ])

def append_client_csv(path: Path, epoch: int, rows: list[dict]):
    with open(path, "a", newline="") as f:
        w = csv.writer(f)
        for r in rows:
            w.writerow([
                int(epoch), int(r["client_id"]), int(r["shard_size"]),
                float(r["latency_sec"]), float(r["qos_mean"]), float(r["qos_var"]),
                float(r["param_var"]), float(r["trust_weight"]),
                float(r["train_acc_local"]), float(r["test_acc_local"]),
            ])
        f.flush()

def save_curve_csv(path: Path, curve: np.ndarray):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["epoch","global_acc"])
        for t, a in enumerate(curve.tolist()):
            w.writerow([t, float(a)])
        f.flush()
        print(f"Saved {path}")


import pandas as pd
import numpy as np

def summarize_final_accuracy(results_dict, out_csv: str):
    rows = []
    for scheme, curves in results_dict.items():  # curves shape (n_seeds, T)
        finals = curves[:, -1]
        n = finals.shape[0]
        mean = float(np.mean(finals))
        std  = float(np.std(finals, ddof=1)) if n > 1 else 0.0
        ci95 = float(1.96 * std / np.sqrt(n)) if n > 1 else 0.0
        rows.append({
            "scheme": scheme,
            "n_seeds": n,
            "final_mean": mean,
            "final_std": std,
            "final_ci95": ci95,
        })
    df = pd.DataFrame(rows).sort_values("final_mean", ascending=False)
    df.to_csv(out_csv, index=False)
    return df

Let's Run FedCompass and FedMRUR baselines
FEDCOMPASS uses client clustering and circular aggregation for quantum parameters in hybrid classical–quantum FL, while FedMRUR uses manifold model fusion and normalized update reaggregation to reduce model inconsistency under heterogeneous FL.

FedMRUR’s key idea relevant to your comparison is normalized update reaggregation. It tries to avoid weak global movement caused by inconsistent or near-orthogonal client updates

FedMRUR-lite controls update norm.
SCM controls whether the movement remains supported from its own midpoint.

In [8]:
from sklearn.cluster import KMeans
import numpy as np

def wrap_pi(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def circular_mean_params(params_list, weights):
    A = np.stack(params_list, axis=0)
    W = np.asarray(weights, dtype=float)
    W = W / (W.sum() + 1e-12)

    sin_mean = np.sum(W[:, None] * np.sin(A), axis=0)
    cos_mean = np.sum(W[:, None] * np.cos(A), axis=0)

    return np.arctan2(sin_mean, cos_mean)

def fedcompass_lite_aggregate(
    local_params,
    data_weights,
    client_signatures,
    n_clusters=2
):
    local_params = [np.asarray(p, dtype=float) for p in local_params]
    data_weights = np.asarray(data_weights, dtype=float)
    data_weights = data_weights / (data_weights.sum() + 1e-12)

    S = np.asarray(client_signatures, dtype=float)

    if len(local_params) < n_clusters:
        return circular_mean_params(local_params, data_weights)

    labels = KMeans(n_clusters=n_clusters, random_state=0, n_init=10).fit_predict(S)

    cluster_params = []
    cluster_weights = []

    for c in range(n_clusters):
        idx = np.where(labels == c)[0]
        if len(idx) == 0:
            continue

        cw = data_weights[idx]
        cw = cw / (cw.sum() + 1e-12)

        cp = circular_mean_params([local_params[i] for i in idx], cw)
        cluster_params.append(cp)
        cluster_weights.append(data_weights[idx].sum())

    cluster_weights = np.asarray(cluster_weights)
    cluster_weights = cluster_weights / (cluster_weights.sum() + 1e-12)

    return circular_mean_params(cluster_params, cluster_weights)

In [9]:
def fedmrur_lite_aggregate(
    global_params,
    local_params,
    weights,
    eta=1.0,
    eps=1e-12
):
    global_params = np.asarray(global_params, dtype=float)
    A = np.stack([np.asarray(p, dtype=float) for p in local_params], axis=0)
    W = np.asarray(weights, dtype=float)
    W = W / (W.sum() + eps)

    updates = A - global_params[None, :]
    norms = np.linalg.norm(updates, axis=1) + eps

    unit_updates = updates / norms[:, None]

    direction = np.sum(W[:, None] * unit_updates, axis=0)
    direction_norm = np.linalg.norm(direction) + eps

    avg_norm = np.sum(W * norms)

    global_update = eta * avg_norm * direction / direction_norm

    return global_params + global_update

def fedmrur_torus_aggregate(
    global_params,
    local_params,
    weights,
    eta=1.0,
    eps=1e-12
):
    global_params = np.asarray(global_params, dtype=float)
    A = np.stack([np.asarray(p, dtype=float) for p in local_params], axis=0)
    W = np.asarray(weights, dtype=float)
    W = W / (W.sum() + eps)

    updates = wrap_pi(A - global_params[None, :])
    norms = np.linalg.norm(updates, axis=1) + eps

    unit_updates = updates / norms[:, None]

    direction = np.sum(W[:, None] * unit_updates, axis=0)
    direction_norm = np.linalg.norm(direction) + eps

    avg_norm = np.sum(W * norms)

    global_update = eta * avg_norm * direction / direction_norm

    return wrap_pi(global_params + global_update)

## 6. Federated training loop with A2G

In [12]:
# ============================================================
# 6. Federated training loop with A2G
# ============================================================

#AGG_SCHEME = "A2G_FULL"   # "BASELINE", "A2G_QOS_ONLY", "A2G_FULL"
num_epochs = 20

QOS_SIGNAL = "quantum_fidelity"   # or "classical_proxy"
GEOMETRY_MODE = "torus_midpoint"       # "euclidean" | "circular" | "torus_midpoint"

# Teleportation config + link (used if QoS enabled)
#tele_cfg = TeleportCfg(use_teleportation=False, noise="med",
                       #shots=256, alpha=0.0, gamma=0.0,
                      # delta=0.0, beta=1.0, seed=2025)
#tele_link = TeleportationLink(tele_cfg)
tele_cfg = TeleportCfg(use_teleportation=True, noise="med", noise_model="depolarizing", shots=256, seed=2025)
tele_link = TeleportationLink(tele_cfg)

from pathlib import Path

# --------------------------------------------------
# High-level experiment identifiers
# --------------------------------------------------


# Combined tag used in all filenames
EXP_TAG = f"{DATASET}"

# Output directory. Works in both Google Colab and local/cluster environments.
try:
    DRIVE_DIR = Path("SCMResults")
except Exception:
    # Local Windows / HPC fallback. Override with environment variable if needed:
    #   set RESULTS_DIR=C:\path\to\results   (Windows cmd)
    #   $env:RESULTS_DIR="C:\path\to\results" (PowerShell)
    DRIVE_DIR = Path(os.getenv("RESULTS_DIR", "./QCNC_A2G_JournalVersionv2"))

DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# Metrics CSV for this dataset + scheme
BASE_CSV_PATH = DRIVE_DIR / f"results_{EXP_TAG}_newupdate.csv"
init_metrics_csv(BASE_CSV_PATH, num_clients)

# Subfolders inside QCNC_A2G
QCNC_CLIENT_DIR = DRIVE_DIR / "QCNC" / "Client"
QCNC_CLIENT_DIR.mkdir(parents=True, exist_ok=True)

# Telemetry files (round + client)
round_csv_path  = QCNC_CLIENT_DIR / f"_round_telemetry_{EXP_TAG}_newupdate.csv"
client_csv_path = QCNC_CLIENT_DIR / f"_client_telemetry_{EXP_TAG}_newupdate.csv"



# CSV paths
#BASE_CSV_PATH = "results_A2G_new.csv"    # change to a Drive path if needed
#init_metrics_csv(BASE_CSV_PATH, num_clients)
#base_path = Path(BASE_CSV_PATH)
#round_csv_path  = base_path.parent / "QCNC" / "Client" / "_round_telemetry_A2GNew.csv"
#client_csv_path = base_path.parent / "QCNC" / "Client" / "_client_telemetry_A2GNew.csv"

global_model_weights_by_epoch = {}
global_model_accuracy = []
clients_train_accuracies = []
clients_test_accuracies = []

# Broadcasted global weights (θ_t); None in epoch 0
global_weights = None

import math
import numpy as np
import random
import torch
import matplotlib.pyplot as plt
from qiskit_algorithms.utils import algorithm_globals

# -----------------------------
# Seed control (important for reviewer)
# -----------------------------
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    algorithm_globals.random_seed = seed
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass

# -----------------------------
# Scheme config (adds easy baselines)
# -----------------------------
# Each scheme sets (alpha,gamma,delta,beta) and which QoS signal to use.
SCHEMES = {
    # --------------------------------------------------------
    # Baselines and conference-version A2G variants
    # --------------------------------------------------------
    "FedAvg": dict(
        qos_signal="none", alpha=0.0, gamma=0.0, delta=0.0,
        beta=1.0, geometry="euclidean",
    ),

    # QoS-only: client weights use fidelity/latency/instability, but no cautious geometry relaxation.
    "QoS_ONLY_Euclidean": dict(
        qos_signal="quantum_fidelity", alpha=1.0, gamma=1.0, delta=1.0,
        beta=1.0, geometry="euclidean",
    ),

    # Conference-style A2G: QoS trust + small beta Euclidean relaxation.
    "A2G_Euclidean": dict(
        qos_signal="quantum_fidelity", alpha=1.0, gamma=1.0, delta=1.0,
        beta=0.05, geometry="euclidean",
    ),

    # Conference-style geometry-aware variant: QoS trust + circular relaxation.
    "A2G_Circular": dict(
        qos_signal="quantum_fidelity", alpha=1.0, gamma=1.0, delta=1.0,
        beta=0.2, geometry="circular",
    ),

    # --------------------------------------------------------
    # Journal-version proposed methods
    # --------------------------------------------------------
    # Fixed-beta midpoint-projected manifold A2G-QFL on T^d.
    "MP_A2G_QFL": dict(
        qos_signal="quantum_fidelity", alpha=1.0, gamma=1.0, delta=1.0,
        beta=0.2, geometry="torus_midpoint",
        adaptive_beta=False, beta_min=0.005, beta_lambda=1.0,
        max_tangent_norm=0.0,
    ),

    # Adaptive-beta midpoint-projected manifold A2G-QFL.
    "Adaptive_MP_A2G_QFL": dict(
        qos_signal="quantum_fidelity", alpha=1.0, gamma=1.0, delta=1.0,
        beta=0.10, geometry="torus_midpoint",
        adaptive_beta=True, beta_min=0.005, beta_lambda=1.0,
        max_tangent_norm=0.0,
    ),

        # Self-Consistent Midpoint A2G-QFL.
    # beta is beta0 in the implicit equation:
    # u* = beta0 * psi(u*)
    "SCM_A2G_QFL": dict(
        qos_signal="quantum_fidelity",
        alpha=1.0,
        gamma=1.0,
        delta=1.0,

        beta=0.05,
        geometry="torus_scm_midpoint",
        adaptive_beta=False,

        beta_min=0.005,
        beta_lambda=1.0,
        max_tangent_norm=0.0,

        scm_iters=5,
        scm_tol=1e-6,
        scm_solver_relax=1.0,
    ),

    # Optional ablation: SCM plus dispersion-adaptive beta0.
    "Adaptive_SCM_A2G_QFL": dict(
        qos_signal="quantum_fidelity",
        alpha=1.0,
        gamma=1.0,
        delta=1.0,
        beta=0.05,
        geometry="torus_scm_midpoint",
        adaptive_beta=True,
        beta_min=0.005,
        beta_lambda=1.0,
        max_tangent_norm=0.0,
        scm_iters=5,
        scm_tol=1e-6,
        scm_solver_relax=1.0,
    ),

     "FEDCOMPASS_lite": dict(
        qos_signal="quantum_fidelity",
        alpha=1.0,
        gamma=1.0,
        delta=1.0,
        beta=1.0,
        geometry="circular",
        aggregator="fedcompass_lite",
        n_clusters=2,
    ),

    "FedMRUR_lite": dict(
        qos_signal="none",
        alpha=0.0,
        gamma=0.0,
        delta=0.0,
        beta=1.0,
        geometry="euclidean",
        aggregator="fedmrur_lite",
        eta=0.15,
    ),

    "FedMRUR_torus": dict(
        qos_signal="quantum_fidelity",
        alpha=1.0,
        gamma=1.0,
        delta=1.0,
        beta=1.0,
        geometry="circular",
        aggregator="fedmrur_torus",
        eta=0.15,
    ),

    # --------------------------------------------------------
    # Extra system/reliability ablations
    # --------------------------------------------------------
    "Latency_ONLY": dict(
        qos_signal="latency_only", alpha=0.0, gamma=1.0, delta=0.0,
        beta=1.0, geometry="euclidean",
    ),

    "A2G_Circular_classical": dict(
        qos_signal="classical_proxy", alpha=1.0, gamma=1.0, delta=1.0,
        beta=0.05, geometry="circular",
    ),

    "Adaptive_MP_A2G_classical": dict(
        qos_signal="classical_proxy", alpha=1.0, gamma=1.0, delta=1.0,
        beta=0.10, geometry="torus_midpoint",
        adaptive_beta=True, beta_min=0.005, beta_lambda=1.0,
        max_tangent_norm=0.0,
    ),

    # Backward-compatible aliases from your previous code.
    "QoS_ONLY": dict(
        qos_signal="quantum_fidelity", alpha=1.0, gamma=1.0, delta=1.0,
        beta=1.0, geometry="euclidean",
    ),
    "A2G_FULL": dict(
        qos_signal="quantum_fidelity", alpha=1.0, gamma=1.0, delta=1.0,
        beta=0.05, geometry="euclidean",
    ),
    "QoS_ONLY_classical": dict(
        qos_signal="classical_proxy", alpha=1.0, gamma=1.0, delta=1.0,
        beta=1.0, geometry="euclidean",
    ),
    "A2G_FULL_classical": dict(
        qos_signal="classical_proxy", alpha=1.0, gamma=1.0, delta=1.0,
        beta=0.05, geometry="euclidean",
    ),
}

# ============================================================
# SCM tuning configurations
# ============================================================

SCM_CONFIGS_TO_TEST = {
    "Journal_SCM_beta003": dict(
        qos_signal="quantum_fidelity",
        alpha=1.0, gamma=1.0, delta=1.0,
        beta=0.03,
        geometry="torus_scm_midpoint",
        adaptive_beta=False,
        scm_iters=5,
        scm_tol=1e-6,
        scm_solver_relax=1.0,
    ),

    "Journal_SCM_beta02": dict(
        qos_signal="quantum_fidelity",
        alpha=1.0, gamma=1.0, delta=1.0,
        beta=0.20,
        geometry="torus_scm_midpoint",
        adaptive_beta=False,
        scm_iters=5,
        scm_tol=1e-6,
        scm_solver_relax=1.0,
    ),

    "Journal_SCM_beta008": dict(
        qos_signal="quantum_fidelity",
        alpha=1.0, gamma=1.0, delta=1.0,
        beta=0.08,
        geometry="torus_scm_midpoint",
        adaptive_beta=False,
        scm_iters=5,
        scm_tol=1e-6,
        scm_solver_relax=1.0,
    ),

    "Journal_SCM_softQoS_beta005": dict(
        qos_signal="quantum_fidelity",
        alpha=1.0, gamma=0.5, delta=0.5,
        beta=0.05,
        geometry="torus_scm_midpoint",
        adaptive_beta=False,
        scm_iters=5,
        scm_tol=1e-6,
        scm_solver_relax=1.0,
    ),

    "Journal_SCM_softQoS_beta008": dict(
        qos_signal="quantum_fidelity",
        alpha=1.0, gamma=0.5, delta=0.5,
        beta=0.08,
        geometry="torus_scm_midpoint",
        adaptive_beta=False,
        scm_iters=5,
        scm_tol=1e-6,
        scm_solver_relax=1.0,
    ),

    # New tuning configurations
    "Journal_SCM_beta010_relax07": dict(
        qos_signal="quantum_fidelity",
        alpha=1.0, gamma=1.0, delta=1.0,
        beta=0.10,
        geometry="torus_scm_midpoint",
        adaptive_beta=False,
        scm_iters=5,
        scm_tol=1e-6,
        scm_solver_relax=0.7,
    ),

    "Journal_SCM_beta015_relax07": dict(
        qos_signal="quantum_fidelity",
        alpha=1.0, gamma=1.0, delta=1.0,
        beta=0.15,
        geometry="torus_scm_midpoint",
        adaptive_beta=False,
        scm_iters=5,
        scm_tol=1e-6,
        scm_solver_relax=0.7,
    ),

    "Journal_SCM_beta020_relax07": dict(
        qos_signal="quantum_fidelity",
        alpha=1.0, gamma=1.0, delta=1.0,
        beta=0.20,
        geometry="torus_scm_midpoint",
        adaptive_beta=False,
        scm_iters=5,
        scm_tol=1e-6,
        scm_solver_relax=0.7,
    ),

    "Journal_SCM_beta020_relax10": dict(
        qos_signal="quantum_fidelity",
        alpha=1.0, gamma=1.0, delta=1.0,
        beta=0.20,
        geometry="torus_scm_midpoint",
        adaptive_beta=False,
        scm_iters=5,
        scm_tol=1e-6,
        scm_solver_relax=1.0,
    ),
}

# Add SCM tuning schemes into the main scheme dictionary
SCHEMES.update(SCM_CONFIGS_TO_TEST)


# ------------------------------------------------------------------
# Explicit paper-comparison names.
# These keep the original conference A2G method and add the journal method
# as separate schemes, so the final tables clearly show old vs new.
# ------------------------------------------------------------------
SCHEMES.update({
    # Current/conference version: QoS trust + direct geometry relaxation.
    "Conf_A2G_Euclidean": SCHEMES["A2G_Euclidean"],
    "Conf_A2G_Circular": SCHEMES["A2G_Circular"],

    # Journal version: QoS trust + midpoint-projected manifold update.
    "Journal_MP_A2G_QFL": SCHEMES["MP_A2G_QFL"],
    "Journal_Adaptive_MP_A2G_QFL": SCHEMES["Adaptive_MP_A2G_QFL"],

    "Journal_SCM_A2G_QFL": SCHEMES["SCM_A2G_QFL"],
    "Journal_Adaptive_SCM_A2G_QFL": SCHEMES["Adaptive_SCM_A2G_QFL"],

})

def compute_global_metrics_from_weights(prototype_model, weights, X, y, eps=1e-12):
    """
    Compute global accuracy and cross-entropy loss from QNN weights.
    Works for binary QNN output with shape (n_samples, 2).
    """
    qnn = getattr(prototype_model, "neural_network", None)
    if qnn is None:
        qnn = getattr(prototype_model, "_neural_network", None)
    if qnn is None:
        raise RuntimeError("Cannot access underlying QNN from classifier.")

    y_raw = qnn.forward(X, np.asarray(weights, float))
    y_raw = np.asarray(y_raw, float)

    # Convert QNN output to probability-like values
    if y_raw.ndim == 1:
        p1 = 1.0 / (1.0 + np.exp(-y_raw))
        probs = np.vstack([1.0 - p1, p1]).T
    else:
        probs = y_raw

    probs = np.clip(probs, eps, 1.0)
    probs = probs / probs.sum(axis=1, keepdims=True)

    y = np.asarray(y).astype(int)
    y_pred = np.argmax(probs, axis=1)

    acc = float(np.mean(y_pred == y))
    loss = float(-np.mean(np.log(probs[np.arange(len(y)), y] + eps)))

    return acc, loss
# -----------------------------
# One seed run
# -----------------------------
from pathlib import Path
import numpy as np

def run_one_seed(
    seed: int,
    scheme_name: str,
    num_epochs: int,
    num_clients: int,
    client_train_data, client_train_labels,
    global_val_data,
    global_test_data,
    tele_cfg_base: TeleportCfg,
    run_dir: Path,
    early_stop=True,
    patience=5,
    min_delta=1e-4,
    update_tol=1e-3,
    scm_residual_tol=1e-4,
):
    set_all_seeds(seed)


    # fresh clients per seed
    clients = [Client((client_train_data[k], client_train_labels[k])) for k in range(num_clients)]
    X_global_val, y_global_val = global_val_data
    X_global_test, y_global_test = global_test_data


    # per-seed teleport link (only used when qos_signal == quantum_fidelity)
    tele_cfg = TeleportCfg(
        use_teleportation=True,
        noise=tele_cfg_base.noise,
        noise_model=tele_cfg_base.noise_model,
        shots=tele_cfg_base.shots,
        seed=seed + 10_000,
    )
    tele_link = TeleportationLink(tele_cfg)

    cfg = SCHEMES[scheme_name]
    geometry_mode = cfg["geometry"]
    qos_signal = cfg["qos_signal"]  # "quantum_fidelity" | "classical_proxy" | "none"

    # files (continuous saving)
    run_dir = Path(run_dir)
    round_csv = run_dir / "round.csv"
    client_csv = run_dir / "client.csv"
    curve_csv = run_dir / "curve.csv"
    init_round_csv(round_csv)
    init_client_csv(client_csv)

    val_csv = run_dir / "validation_curve.csv"

    with open(val_csv, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow([
            "epoch",
            "global_acc",
            "val_acc",
            "val_loss",
            "test_acc_current",
            "test_loss_current",
            "best_val_loss",
            "best_val_acc",
            "best_epoch",
            "no_improve_count",
            "update_norm",
            "scm_residual",
            "stationary_flag",
        ])
        

    global_acc_curve = []
    num_features = client_train_data[0].shape[1]

    tmp_model = create_qnn_model(num_features)
    num_weights = tmp_model.neural_network.num_weights

    rng = np.random.default_rng(seed)
    global_weights = rng.uniform(-0.1, 0.1, size=num_weights)

    best_val_loss = np.inf
    best_val_acc = -np.inf
    best_epoch = -1
    best_global_weights = None
    best_diag = None
    no_improve_count = 0
    stationary_epoch = None
    min_delta = 1e-4

    for epoch in range(num_epochs):
        epoch_weights = []
        shard_sizes = []
        latencies = []
        param_vars = []
        train_accs = []
        test_accs = []

        # store both signals (for transparency)
        F_means, F_vars = [], []
        R_means, R_vars = [], []

        print(f"\n[RUN] scheme={scheme_name} | seed={seed} | epochs={num_epochs} | dir={run_dir}")

        # --- Local training per client ---
        # --- Local training per client ---
        for cid, client in enumerate(clients):
            X_client_train, y_client_train = client.client_train_data

            # Create a fresh client model for this round,
            # starting exactly from the server global model.
            model = create_qnn_model(
                num_features=X_client_train.shape[1],
                initial_point=global_weights
            )

            model, train_score, test_score, t_local = train_qnn_model(
                X_client_train,
                y_client_train,
                X_global_test,
                y_global_test,
                model=model,
                eval_local=False,
            )

            client.primary_model = model

            w_local = extract_client_weights(model)
            epoch_weights.append(np.asarray(w_local, float))
            shard_sizes.append(len(X_client_train))
            latencies.append(max(t_local, 1e-6))
            param_vars.append(param_variance(w_local))
            train_accs.append(float(train_score))
            test_accs.append(float(test_score))

            # quantum fidelity samples ONLY if this scheme uses them
            if qos_signal == "quantum_fidelity":
                F_samps = tele_link.sample_fidelity()
                F_means.append(float(np.mean(F_samps)))
                F_vars.append(float(np.var(F_samps)))
            else:
                F_means.append(1.0)
                F_vars.append(0.0)

        epoch_weights = [np.asarray(w, float) for w in epoch_weights]
        latencies = np.asarray(latencies, float)

        # classical proxy R based on latency (ONLY relevant if qos_signal == classical_proxy)
        rng_proxy = np.random.default_rng(seed + 999 + epoch)
        L_ref = float(np.median(latencies) + 1e-6)
        for L_i in latencies:
            r_mean, r_var = sample_packet_success_proxy(
                L_i, shots=tele_cfg.shots, rng=rng_proxy, L_ref=L_ref
            )
            R_means.append(r_mean)
            R_vars.append(r_var)
        R_means = np.asarray(R_means, float)
        R_vars  = np.asarray(R_vars, float)

        F_means = np.asarray(F_means, float)
        F_vars  = np.asarray(F_vars, float)

        # choose QoS signal used in weighting + logging
        if qos_signal == "quantum_fidelity":
            qos_means = F_means
            qos_vars  = F_vars
            lat_use   = latencies
        elif qos_signal == "classical_proxy":
            qos_means = R_means
            qos_vars  = R_vars
            lat_use   = latencies
        elif qos_signal == "latency_only":
        # use Q=1, V=0, but KEEP latency in the denominator via gamma
            K = len(clients)
            qos_means = np.ones(K)
            qos_vars  = np.zeros(K)
            lat_use   = latencies
        else:
            # FedAvg: QoS cancels out (set to constants)
            K = len(clients)
            qos_means = np.ones(K)
            qos_vars  = np.zeros(K)
            lat_use   = np.ones(K)

        # aggregator config
        tele_cfg_agg = TeleportCfg(
            use_teleportation=(qos_signal != "none"),
            noise=tele_cfg.noise,
            noise_model=tele_cfg.noise_model,
            shots=tele_cfg.shots,
            alpha=cfg["alpha"],
            gamma=cfg["gamma"],
            delta=cfg["delta"],
            beta=cfg["beta"],
            adaptive_beta=cfg.get("adaptive_beta", False),
            beta_min=cfg.get("beta_min", 0.005),
            beta_lambda=cfg.get("beta_lambda", 1.0),
            max_tangent_norm=cfg.get("max_tangent_norm", 0.0),
            seed=seed + 20_000,
        )
                # Extra attributes for SCM geometry.
        # This avoids changing the TeleportCfg dataclass.
        tele_cfg_agg.scm_iters = cfg.get("scm_iters", 5)
        tele_cfg_agg.scm_tol = cfg.get("scm_tol", 1e-6)
        tele_cfg_agg.scm_solver_relax = cfg.get("scm_solver_relax", 1.0)

        # --- aggregation ---
        average_weights, diag, agg_w = agg_a2g_qos(
            local_ws=epoch_weights,
            shard_sizes=shard_sizes,
            quality_means=qos_means,
            latencies=lat_use,
            quality_vars=qos_vars,
            tele_cfg=tele_cfg_agg,
            prev_global=global_weights,
            geometry=geometry_mode,
        )
        global_weights = np.asarray(average_weights, float)

        # Save global parameters for this round
        params_dir = run_dir / "global_params"
        params_dir.mkdir(parents=True, exist_ok=True)

        save_global_params_npz(
            path=params_dir / f"global_params_round_{epoch:03d}.npz",
            global_weights=global_weights,
            epoch=epoch,
            scheme_name=scheme_name,
            seed=seed,
            dataset_name=DATASET,
            geometry=geometry_mode,
            qos_signal=qos_signal,
            diag=diag,
        )

        # --- global evaluation ---
        prototype = clients[0].primary_model
        global_acc = compute_global_accuracy_from_weights(
            prototype, global_weights, X_global_test, y_global_test
        )
        global_acc_curve.append(float(global_acc))

        # --- validation and current test metrics ---
        val_acc, val_loss = compute_global_metrics_from_weights(
            prototype,
            global_weights,
            X_global_val,
            y_global_val,
        )

        test_acc_current, test_loss_current = compute_global_metrics_from_weights(
            prototype,
            global_weights,
            X_global_test,
            y_global_test,
        )

        # --- best checkpoint selection using validation loss ---
        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_epoch = epoch
            best_global_weights = global_weights.copy()
            best_diag = dict(diag)
            no_improve_count = 0
        else:
            no_improve_count += 1

        # --- stationary condition ---
        update_norm = float(diag.get("update_norm", np.nan))
        scm_residual = float(diag.get("scm_residual", np.nan))

        small_update = np.isfinite(update_norm) and update_norm < update_tol

        if np.isnan(scm_residual):
            small_scm_residual = True
        else:
            small_scm_residual = scm_residual < scm_residual_tol

        loss_plateau = no_improve_count >= patience

        stationary_flag = int(loss_plateau and small_update and small_scm_residual)

        if stationary_flag and stationary_epoch is None:
            stationary_epoch = epoch

        # --- save validation curve ---
        with open(val_csv, "a", newline="") as f:
            w = csv.writer(f)
            w.writerow([
                int(epoch),
                float(global_acc),
                float(val_acc),
                float(val_loss),
                float(test_acc_current),
                float(test_loss_current),
                float(best_val_loss),
                float(best_val_acc),
                int(best_epoch),
                int(no_improve_count),
                float(update_norm),
                float(scm_residual) if np.isfinite(scm_residual) else np.nan,
                int(stationary_flag),
            ])

        print(
            f"  [VAL] epoch={epoch:02d} | val_acc={val_acc:.4f} | "
            f"val_loss={val_loss:.6f} | best_epoch={best_epoch} | "
            f"no_improve={no_improve_count} | update_norm={update_norm:.6f}"
        )

        # if early_stop and loss_plateau:
        #     print(
        #         f"[EARLY STOP] seed={seed}, scheme={scheme_name}: "
        #         f"validation loss did not improve for {patience} rounds. "
        #         f"Best epoch={best_epoch}, best_val_loss={best_val_loss:.6f}"
        #     )
        #     break

        # --- continuous saving (round-wise + client-wise) ---
        append_round_csv(
            path=round_csv,
            epoch=epoch,
            global_acc=global_acc,
            qos_means=qos_means,
            latencies=latencies,
            qos_vars=qos_vars,
            agg_w=agg_w,
            diag=diag,
            qos_signal=qos_signal,
            noise=tele_cfg.noise,
            noise_model=tele_cfg.noise_model,
        )

        client_rows = []
        for cid in range(num_clients):
            client_rows.append({
                "client_id": cid,
                "shard_size": shard_sizes[cid],
                "latency_sec": float(latencies[cid]),
                "qos_mean": float(qos_means[cid]),
                "qos_var": float(qos_vars[cid]),
                "param_var": float(param_vars[cid]),
                "trust_weight": float(agg_w[cid]),
                "train_acc_local": float(train_accs[cid]),
                "test_acc_local": float(test_accs[cid]),
            })
        append_client_csv(client_csv, epoch, client_rows)


        if (epoch % 1 == 0) or (epoch == num_epochs - 1):
          print(f"  [seed {seed}] epoch {epoch+1:02d}/{num_epochs} | global_acc={global_acc:.4f} "
          f"| qos_mean={float(np.mean(qos_means)):.3f} | lat_mean={float(np.mean(latencies)):.3f}")

    # save per-seed curve
    save_curve_csv(curve_csv, np.asarray(global_acc_curve, float))
    save_global_params_npz(
    path=run_dir / "global_params_final.npz",
    global_weights=global_weights,
    epoch=num_epochs - 1,
    scheme_name=scheme_name,
    seed=seed,
    dataset_name=DATASET,
    geometry=geometry_mode,
    qos_signal=qos_signal,
    diag=diag,
)
    
    if best_global_weights is None:
        selected_global_weights = global_weights.copy()
        selected_epoch = epoch
        selected_diag = dict(diag)
    else:
        selected_global_weights = best_global_weights.copy()
        selected_epoch = best_epoch
        selected_diag = best_diag if best_diag is not None else dict(diag)

    selected_test_acc, selected_test_loss = compute_global_metrics_from_weights(
        prototype,
        selected_global_weights,
        X_global_test,
        y_global_test,
    )

    save_global_params_npz(
        path=run_dir / "global_params_selected_best_val.npz",
        global_weights=selected_global_weights,
        epoch=selected_epoch,
        scheme_name=scheme_name,
        seed=seed,
        dataset_name=DATASET,
        geometry=geometry_mode,
        qos_signal=qos_signal,
        diag=selected_diag,
    )

    with open(run_dir / "selected_checkpoint_summary.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow([
            "scheme",
            "seed",
            "selected_epoch",
            "best_val_acc",
            "best_val_loss",
            "selected_test_acc",
            "selected_test_loss",
            "final_epoch",
        ])
        w.writerow([
            scheme_name,
            seed,
            selected_epoch,
            best_val_acc,
            best_val_loss,
            selected_test_acc,
            selected_test_loss,
            epoch,
        ])

    print(
        f"[SELECTED] scheme={scheme_name} | seed={seed} | "
        f"selected_epoch={selected_epoch} | best_val_loss={best_val_loss:.6f} | "
        f"test_acc={selected_test_acc:.4f} | test_loss={selected_test_loss:.6f}"
    )
    return np.asarray(global_acc_curve, float)
# -----------------------------
# Mean ± 95% CI plot
# -----------------------------
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import csv
import os

def tcrit_95(n: int) -> float:
    """Two-sided 95% t critical value (small n). Fallback to 1.96."""
    table = {
        2: 12.706, 3: 4.303, 4: 3.182, 5: 2.776,
        6: 2.571, 7: 2.447, 8: 2.365, 9: 2.306, 10: 2.262
    }
    return table.get(int(n), 1.96)

def mean_ci95(curves: np.ndarray):
    """
    curves: (n_seeds, T)
    returns mean(T), lo(T), hi(T)
    """
    curves = np.asarray(curves, float)
    n = curves.shape[0]
    mu = curves.mean(axis=0)
    if n <= 1:
        return mu, mu, mu
    sd = curves.std(axis=0, ddof=1)
    sem = sd / np.sqrt(n)
    tc = tcrit_95(n)
    lo = mu - tc * sem
    hi = mu + tc * sem
    return mu, lo, hi

def plot_schemes_with_ci(results_dict: dict, title: str, save_path: str):
    """
    results_dict: {scheme_name: curves (n_seeds, T)}
    """
    plt.figure()
    for scheme, curves in results_dict.items():
        mu, lo, hi = mean_ci95(curves)
        x = np.arange(len(mu))
        plt.plot(x, mu, label=f"{scheme} (n={curves.shape[0]})")
        plt.fill_between(x, lo, hi, alpha=0.2)

    plt.xlabel("Epoch")
    plt.ylabel("Global accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    save_path = str(save_path)
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()

def save_final_table(results_dict: dict, csv_path: str):
    """
    Save final-epoch accuracy mean ± 95% CI for each scheme.
    """
    rows = []
    for scheme, curves in results_dict.items():
        curves = np.asarray(curves, float)
        n = curves.shape[0]
        final = curves[:, -1]
        mu = float(final.mean())
        if n > 1:
            sd = float(final.std(ddof=1))
            sem = sd / np.sqrt(n)
            tc = tcrit_95(n)
            ci = float(tc * sem)
        else:
            ci = 0.0
        rows.append((scheme, n, mu, ci))

    rows.sort(key=lambda x: x[2], reverse=True)

    csv_path = str(csv_path)
    Path(csv_path).parent.mkdir(parents=True, exist_ok=True)
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["scheme", "n_seeds", "final_mean", "final_ci95"])
        for r in rows:
            w.writerow(list(r))

def save_global_params_npz(
    path: Path,
    global_weights: np.ndarray,
    epoch: int,
    scheme_name: str,
    seed: int,
    dataset_name: str,
    geometry: str,
    qos_signal: str,
    diag: dict,
):
    """
    Save global QNN parameters and metadata for later IBM hardware testing.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    np.savez(
        path,
        global_weights=np.asarray(global_weights, dtype=float),
        epoch=int(epoch),
        scheme_name=str(scheme_name),
        seed=int(seed),
        dataset_name=str(dataset_name),
        geometry=str(geometry),
        qos_signal=str(qos_signal),
        beta=float(diag.get("beta", np.nan)),
        beta_eff=float(diag.get("beta_eff", diag.get("beta", np.nan))),
        manifold_dispersion=float(diag.get("manifold_dispersion", np.nan)),
        tangent_norm=float(diag.get("tangent_norm", np.nan)),
        update_norm=float(diag.get("update_norm", np.nan)),
    )
# -----------------------------
# Run ≥5 seeds
# -----------------------------
from pathlib import Path
import numpy as np

# -----------------------------
# Run ≥5 seeds
# -----------------------------
N_SEEDS = 5   # set 10 if time allows
SEEDS = [100, 200, 300, 400, 500][:N_SEEDS]

NUM_EPOCHS = num_epochs
tele_cfg_base = tele_cfg

# If  don't already have these lists (for sk_breast/genome), derive them from `clients`
if "client_train_data" not in globals():
    client_train_data = [clients[k].client_train_data[0] for k in range(num_clients)]
    client_train_labels = [clients[k].client_train_data[1] for k in range(num_clients)]

SCHEMES_TO_RUN = [  
    #"Journal_SCM_A2G_QFL",
    #"Journal_MP_A2G_QFL",
    #"Conf_A2G_Circular",
    #"FedAvg",
    #"Journal_Adaptive_SCM_A2G_QFL",
]
SCHEMES_TO_RUN = [
    # "Journal_SCM_beta010_relax07",
    # "Journal_SCM_beta015_relax07",
    # "Journal_SCM_beta020_relax07",
    # "Journal_SCM_beta020_relax10",
    # "Journal_SCM_beta02",
    # "Journal_SCM_beta003",
    # "Journal_SCM_beta008",
    # "Journal_SCM_softQoS_beta005",
    # "Journal_SCM_softQoS_beta008",
    #"FedAvg",
    #"Conf_A2G_Circular",
    #"Conf_A2G_Euclidean",
    #"FEDCOMPASS_lite",
    "FedMRUR_torus",
    



]
# SCHEMES_TO_RUN = [
#     # Baselines
#     #"FedAvg",
#     #"QoS_ONLY_Euclidean",

#     # Current conference-version A2G methods
#     #"Conf_A2G_Euclidean",
#     #"Conf_A2G_Circular",

#     # Proposed journal-version methods
#     "Journal_MP_A2G_QFL",
#     "Journal_Adaptive_MP_A2G_QFL",

#     # Optional ablations:
#     # "Latency_ONLY",
#     # "A2G_Circular_classical",
#     # "Adaptive_MP_A2G_classical",
# ]
# SCHEMES_TO_RUN = [
#     "FedAvg",
#     "Conf_A2G_Euclidean",
#     "Conf_A2G_Circular",
#     "Journal_MP_A2G_QFL",
#     "Journal_Adaptive_MP_A2G_QFL",
#     "Journal_SCM_A2G_QFL",
#     "Journal_Adaptive_SCM_A2G_QFL",
# ]

results = {}

RUNS_DIR = Path(DRIVE_DIR) / "runs" / f"{DATASET}_{tele_cfg_base.noise}_{tele_cfg_base.noise_model}"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

for scheme_name in SCHEMES_TO_RUN:
    all_curves = []
    print(f"\n=== Running scheme: {scheme_name} ===")
    for s in SEEDS:
        print(f" -> seed {s} ...")

        run_dir = RUNS_DIR / scheme_name / f"seed_{s}"
        run_dir.mkdir(parents=True, exist_ok=True)

        curve = run_one_seed(
            seed=s,
            scheme_name=scheme_name,
            num_epochs=NUM_EPOCHS,
            num_clients=num_clients,
            client_train_data=client_train_data,
            client_train_labels=client_train_labels,
            global_val_data=global_val_data,
            global_test_data=global_test_data,
            tele_cfg_base=tele_cfg_base,
            run_dir=run_dir,
            early_stop=True,
            patience=5,
            min_delta=1e-4,
            update_tol=1e-3,
            scm_residual_tol=1e-4,
        )
        all_curves.append(curve)

    results[scheme_name] = np.stack(all_curves, axis=0)  # (n_seeds, T)

# Plot mean ± 95% CI
plot_path = RUNS_DIR / f"plot_meanCI_{DATASET}_{tele_cfg_base.noise}_{tele_cfg_base.noise_model}_n{N_SEEDS}.png"
plot_schemes_with_ci(
    results_dict=results,
    title=f"{DATASET} | mean ± 95% CI | {tele_cfg_base.noise}/{tele_cfg_base.noise_model} | n={N_SEEDS}",
    save_path=str(plot_path),
)

# Save final table (camera-ready)
table_path = RUNS_DIR / f"final_table_{DATASET}_{tele_cfg_base.noise}_{tele_cfg_base.noise_model}_n{N_SEEDS}.csv"
save_final_table(results, str(table_path))

print("Saved plot:", plot_path)
print("Saved final table:", table_path)
print("Saved per-seed logs under:", RUNS_DIR)


=== Running scheme: FedMRUR_torus ===
 -> seed 100 ...

[RUN] scheme=FedMRUR_torus | seed=100 | epochs=20 | dir=SCMResults\runs\lesions_med_depolarizing\FedMRUR_torus\seed_100
  [VAL] epoch=00 | val_acc=0.3871 | val_loss=0.760344 | best_epoch=0 | no_improve=0 | update_norm=4.820818
  [seed 100] epoch 01/20 | global_acc=0.4688 | qos_mean=0.971 | lat_mean=32.802

[RUN] scheme=FedMRUR_torus | seed=100 | epochs=20 | dir=SCMResults\runs\lesions_med_depolarizing\FedMRUR_torus\seed_100
  [VAL] epoch=01 | val_acc=0.2903 | val_loss=0.757376 | best_epoch=1 | no_improve=0 | update_norm=2.359491
  [seed 100] epoch 02/20 | global_acc=0.5000 | qos_mean=0.969 | lat_mean=47.714

[RUN] scheme=FedMRUR_torus | seed=100 | epochs=20 | dir=SCMResults\runs\lesions_med_depolarizing\FedMRUR_torus\seed_100
  [VAL] epoch=02 | val_acc=0.5161 | val_loss=0.715164 | best_epoch=2 | no_improve=0 | update_norm=2.291840
  [seed 100] epoch 03/20 | global_acc=0.5000 | qos_mean=0.970 | lat_mean=30.923

[RUN] scheme=FedMRU

In [ ]:
def plot_validation_loss_for_run(run_dir: Path):
    val_csv = Path(run_dir) / "validation_curve.csv"
    if not val_csv.exists():
        print(f"Missing validation curve: {val_csv}")
        return

    df = pd.read_csv(val_csv)

    best_idx = int(df["val_loss"].idxmin())
    best_epoch = int(df.loc[best_idx, "epoch"])
    best_loss = float(df.loc[best_idx, "val_loss"])

    stationary_rows = df[df["stationary_flag"] == 1]
    if len(stationary_rows) > 0:
        stationary_epoch = int(stationary_rows["epoch"].iloc[0])
    else:
        stationary_epoch = None

    plt.figure(figsize=(7, 4))
    plt.plot(df["epoch"], df["val_loss"], marker="o", label="Validation loss")

    plt.scatter(
        [best_epoch],
        [best_loss],
        s=80,
        label=f"Best checkpoint: epoch {best_epoch}",
    )

    if stationary_epoch is not None:
        stat_loss = float(df.loc[df["epoch"] == stationary_epoch, "val_loss"].iloc[0])
        plt.axvline(
            stationary_epoch,
            linestyle="--",
            label=f"Stationary/early-stop epoch {stationary_epoch}",
        )
        plt.scatter([stationary_epoch], [stat_loss], s=80)

    plt.xlabel("Federated round")
    plt.ylabel("Validation loss")
    plt.title(f"Validation loss curve\n{Path(run_dir).parent.name} / {Path(run_dir).name}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    out_path = Path(run_dir) / "validation_loss_best_stationary.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

    print("Saved:", out_path)

In [ ]:
plot_validation_loss_for_run(
    Path(r"SCMResults\runs\lesions_med_depolarizing\Journal_SCM_beta010_relax07\seed_100")
)

In [ ]:
def plot_validation_accuracy_for_run(run_dir: Path):
    val_csv = Path(run_dir) / "validation_curve.csv"
    if not val_csv.exists():
        print(f"Missing validation curve: {val_csv}")
        return

    df = pd.read_csv(val_csv)

    best_idx = int(df["val_loss"].idxmin())
    best_epoch = int(df.loc[best_idx, "epoch"])

    plt.figure(figsize=(7, 4))
    plt.plot(df["epoch"], df["val_acc"], marker="o", label="Validation accuracy")
    plt.plot(df["epoch"], df["test_acc_current"], marker="s", label="Current test accuracy")

    plt.axvline(best_epoch, linestyle="--", label=f"Best val-loss epoch {best_epoch}")

    plt.xlabel("Federated round")
    plt.ylabel("Accuracy")
    plt.title(f"Accuracy curve\n{Path(run_dir).parent.name} / {Path(run_dir).name}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    out_path = Path(run_dir) / "validation_accuracy_best_checkpoint.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

    print("Saved:", out_path)

## After running

The notebook saves per-method/per-seed files under the configured `RUNS_DIR`:

- `round.csv`: global accuracy and round-level QoS/manifold telemetry
- `client.csv`: client-level telemetry
- `curve.csv`: global accuracy curve
- final comparison CSV table
- mean ± 95% CI accuracy plot

For the journal paper, the most important comparison is:

```text
Conf_A2G_Circular
vs
Journal_MP_A2G_QFL
vs
Journal_Adaptive_MP_A2G_QFL
```
